In [1]:
from google.colab import files

uploaded = files.upload()

Saving Fact_Trades.csv to Fact_Trades.csv


In [11]:
import pandas as pd

trades = pd.read_csv(
    "Fact_Trades.csv",
    sep=","
)

trades.head()

,Trade_ID,Entry_Time,Exit_Time,Entry_Price,Exit_Price,PnL,Holding_Hours,Strategy_ID
0,1,2023-01-03 23:00:00,2023-01-04 17:00:00,93.66,86.16,-7.50,18.0,1
1,2,2023-01-04 23:00:00,2023-01-05 06:00:00,9.09,61.04,51.95,7.0,1
2,3,2023-01-07 04:00:00,2023-01-08 04:00:00,82.99,5.58,-77.41,24.0,1
3,4,2023-01-10 21:00:00,2023-01-11 08:00:00,86.53,127.44,40.91,11.0,1
4,5,2023-01-12 01:00:00,2023-01-12 07:00:00,5.55,111.55,106.00,6.0,1


In [21]:
trades["Entry_Time"] = pd.to_datetime(trades["Entry_Time"]).dt.floor("D")
trades["Exit_Time"] = pd.to_datetime(trades["Exit_Time"]).dt.floor("D")

trades.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 174 entries, 0 to 173
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Trade_ID       174 non-null    int64         
 1   Entry_Time     174 non-null    datetime64[ns]
 2   Exit_Time      174 non-null    datetime64[ns]
 3   Entry_Price    174 non-null    float64       
 4   Exit_Price     174 non-null    float64       
 5   PnL            174 non-null    float64       
 6   Holding_Hours  174 non-null    float64       
 7   Strategy_ID    174 non-null    int64         
dtypes: datetime64[ns](2), float64(4), int64(2)
memory usage: 11.0 KB


In [22]:
#Vreate daily calendar
calendar = pd.DataFrame({
    "Date": pd.date_range(
        start="2023-01-01",
        end="2023-12-31",
        freq="D"
    )
})

calendar.head()

,Date
0,2023-01-01
1,2023-01-02
2,2023-01-03
3,2023-01-04
4,2023-01-05


In [24]:
#Daily PnL
daily_pnl = (
    trades
    .groupby("Entry_Time")["PnL"]
    .sum()
    .reset_index()
)

daily_pnl.columns = ["Date","Daily_PnL"]

daily_pnl.head()

,Date,Daily_PnL
0,2023-01-03,-7.50
1,2023-01-04,51.95
2,2023-01-07,-77.41
3,2023-01-10,40.91
4,2023-01-12,106.00


In [25]:
#Merge Calendar
risk = calendar.merge(
    daily_pnl,
    on="Date",
    how="left"
)

risk["Daily_PnL"] = risk["Daily_PnL"].fillna(0)

risk.head()

,Date,Daily_PnL
0,2023-01-01,0.00
1,2023-01-02,0.00
2,2023-01-03,-7.50
3,2023-01-04,51.95
4,2023-01-05,0.00


In [26]:
#Equity Curve
risk["Cumulative_PnL"] = risk["Daily_PnL"].cumsum()

risk.head()

,Date,Daily_PnL,Cumulative_PnL
0,2023-01-01,0.00,0.00
1,2023-01-02,0.00,0.00
2,2023-01-03,-7.50,-7.50
3,2023-01-04,51.95,44.45
4,2023-01-05,0.00,44.45


In [27]:
#Running Peak
risk["Running_Peak"] = risk["Cumulative_PnL"].cummax()

In [28]:
#Drawdown
risk["Drawdown"] = (
    risk["Cumulative_PnL"]
    -
    risk["Running_Peak"]
)

In [29]:
#Rolling Volatility(30 days)
risk["Rolling_Volatility"] = (
    risk["Daily_PnL"]
    .rolling(30)
    .std()
)

In [30]:
#Rolling Mean
risk["Rolling_Mean"] = (
    risk["Daily_PnL"]
    .rolling(30)
    .mean()
)

In [32]:
#Rolling Sharpe
import numpy as np

risk["Rolling_Sharpe"] = (
    risk["Rolling_Mean"]
    /
    risk["Rolling_Volatility"]
) * np.sqrt(252)

In [33]:
#VaR
risk["VaR95"] = (
    risk["Daily_PnL"]
    .rolling(30)
    .quantile(0.05)
)

In [34]:
#Monthly PnL
risk["Month"] = risk["Date"].dt.to_period("M").astype(str)

monthly = (
    risk.groupby("Month")["Daily_PnL"]
    .sum()
    .reset_index()
)

monthly.head()

,Month,Daily_PnL
0,2023-01,513.56
1,2023-02,262.11
2,2023-03,561.54
3,2023-04,712.43
4,2023-05,1205.95


In [35]:
#Summary Statistics
print("Total PnL:", round(risk["Daily_PnL"].sum(),2))
print("Max Drawdown:", round(risk["Drawdown"].min(),2))
print("Worst Daily PnL:", round(risk["Daily_PnL"].min(),2))
print("Best Daily PnL:", round(risk["Daily_PnL"].max(),2))
print("Average Daily PnL:", round(risk["Daily_PnL"].mean(),2))

Total PnL: 7632.99
Max Drawdown: -77.41
Worst Daily PnL: -77.41
Best Daily PnL: 202.28
Average Daily PnL: 20.91


In [36]:
risk.to_csv(
    "Fact_Risk.csv",
    index=False
)

monthly.to_csv(
    "Monthly_PnL.csv",
    index=False
)